# Phase 7 - Cross-Model Transfer

Este notebook organiza a analise de transferencia: uma direcao latente extraida em um modelo e testada em outro. A pergunta academica e se o sinal parece especifico do modelo ou parcialmente compartilhado.

Importamos bibliotecas e carregamos summaries de steering. O notebook aceita a ausencia de runs cross-model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
SEARCH_DIRS = [ROOT / 'runs' / 'phase4', ROOT / 'runs' / 'phase7']
SEARCH_DIRS

Extraimos modelo, camada, alpha e tipo de direcao. Em experimentos reais, use um diretorio separado para cada modelo alvo.

In [ ]:
rows = []
for directory in SEARCH_DIRS:
    for path in sorted(directory.rglob('*_summary.json')) if directory.exists() else []:
        data = json.loads(path.read_text(encoding='utf-8'))
        steering = data.get('steering', {})
        rows.append({
            'run': path.stem.replace('_summary', ''),
            'model_id': steering.get('model_id'),
            'benchmark': steering.get('benchmark'),
            'layer': steering.get('layer'),
            'alpha': steering.get('alpha'),
            'direction_type': steering.get('direction_type'),
            'directions_dir': steering.get('directions_dir'),
            'accuracy': data.get('observed_best_of_n'),
            'tokens_to_success': data.get('mean_tokens_until_success_or_budget'),
        })

transfer = pd.DataFrame(rows)
transfer

A matriz abaixo facilita comparar o mesmo controle em diferentes modelos. Valores ausentes indicam que a combinacao ainda nao foi rodada.

In [ ]:
if transfer.empty:
    print('Sem resultados cross-model ainda.')
else:
    pivot = transfer.pivot_table(index=['layer', 'alpha', 'direction_type'], columns='model_id', values='accuracy', aggfunc='mean')
    display(pivot)


Esta visualizacao procura estabilidade qualitativa: a direcao correta deveria superar controles em mais de um modelo para sustentar transferencia.

In [ ]:
if transfer.empty:
    print('Rode sweeps em mais de um modelo para visualizar transferencia.')
else:
    ax = transfer.groupby(['model_id', 'direction_type'])['accuracy'].mean().unstack().plot(kind='bar')
    ax.set_ylabel('Acuracia media')
    ax.set_title('Transferencia por modelo e controle')
    plt.show()

Desenho sugerido: extraia direcoes no Qwen2.5-Coder-1.5B, teste no Qwen2.5-Coder-0.5B e em um coder SLM alternativo. Compare sempre com `random_direction` e `negative_correctness_direction`.